In [40]:
import pandas as pd
import sqlite3

# 1. 初始化干净的宇宙
if 'conn' in locals():
    conn.close()
conn = sqlite3.connect(':memory:')

# 🛰️ 核心流水线：交易日志表（Transaction Log）
# 【陷阱】：
# 1. transaction_time 是纯文本字符串（'O' 类型），两端带有恶意空格。
# 2. 701号和702号的数据在数仓里是乱序交错存储的。
tx_data = {
    'tx_id':            [5001,  5002,  5003,  5004,  5005,  5006],
    'user_id':          [701,   702,   701,   701,   702,   702],
    'location':         ['Beijing', 'Shanghai', 'Beijing', 'New_York', 'Shanghai', 'Tokyo'],
    'transaction_time': [
        ' 2026-06-10 12:00:00 ', # 701 在北京交易
        ' 2026-06-10 12:05:00 ', # 702 在上海交易
        ' 2026-06-10 12:10:00 ', # 701 在北京交易（与上一次地点相同，安全）
        ' 2026-06-10 12:15:00 ', # 701 在纽约交易！【🔥 异地突变！】距离上一次过了 5 分钟
        ' 2026-06-10 12:20:00 ', # 702 在上海交易（与上一次地点相同，安全）
        ' 2026-06-10 12:50:00 '  # 702 在东京交易！【🔥 异地突变！】距离上一次过了 30 分钟
    ]
}
df_tx = pd.DataFrame(tx_data)
df_tx.to_sql('transaction_log', conn, index=False, if_exists='replace')

print("=== 🪐 实时反欺诈多维时空沙盒已就绪，等待神罚降临 ===")
print(df_tx)

=== 🪐 实时反欺诈多维时空沙盒已就绪，等待神罚降临 ===
   tx_id  user_id  location       transaction_time
0   5001      701   Beijing   2026-06-10 12:00:00 
1   5002      702  Shanghai   2026-06-10 12:05:00 
2   5003      701   Beijing   2026-06-10 12:10:00 
3   5004      701  New_York   2026-06-10 12:15:00 
4   5005      702  Shanghai   2026-06-10 12:20:00 
5   5006      702     Tokyo   2026-06-10 12:50:00 


### 请分别使用 SQL 轨道（窗口函数） 和 Pandas 轨道（游标位移），完成对这次黑产盗刷的特征提取。

 👑 任务交付标准：
 
输出一张最终的风控审计报表，必须包含且仅包含以下四个字段，并按 user_id 升序、tx_id 升序排列：

* user_id：用户唯一标识。

* tx_id：触发了“异地突变”的第二次交易 ID（在 701 号的例子里，指的就是 5004 这笔纽约交易）。

* current_location：当前这次突变交易的地点（如 New_York）。

* seconds_since_last_tx：当前突变交易时间，距离该用户“紧邻的上一次交易时间”过去了多少秒？（提示：由于是文本时间，需要转换为时间差并提取出总秒数）。

In [41]:
# ====================================================
# 🧱 SQL 轨道 - 单表高维时空互搏完全体（反欺诈特征提取）
# ====================================================
sql_query = """
WITH tx_with_ghost_columns AS (
    SELECT 
        user_id,
        tx_id,
        TRIM(location) AS current_location,
        -- 🛰️ 物理清洗：先削掉时间的恶意空格，熔炼为纯净时间
        datetime(TRIM(transaction_time)) AS current_tx_time,
        
        -- 👻 幽灵广播列一：利用 LAG 抓出“紧邻的上一次交易地点”
        LAG(TRIM(location)) OVER (
            PARTITION BY user_id 
            ORDER BY datetime(TRIM(transaction_time)) ASC
        ) AS last_location,
        
        -- 👻 幽灵广播列二：利用 LAG 抓出“紧邻的上一次交易时间”
        LAG(datetime(TRIM(transaction_time))) OVER (
            PARTITION BY user_id 
            ORDER BY datetime(TRIM(transaction_time)) ASC
        ) AS last_tx_time
    FROM transaction_log
)
SELECT 
    user_id,
    tx_id,
    current_location,
    -- ⚡ 终极熔炼：当前秒戳 - 上一次秒戳 = 纯正的数字秒差
    ( strftime('%s', current_tx_time) - strftime('%s', last_tx_time) ) AS seconds_since_last_tx
FROM tx_with_ghost_columns
-- ⚔️ 终极反欺诈天网：
-- 1. 地点必须发生突变（当前地点 != 上一次地点）
-- 2. 必须排除天字第一号行(第一行往回看没有历史,last_location IS NULL,必须过滤掉)
WHERE current_location <> last_location 
  AND last_location IS NOT NULL
ORDER BY user_id ASC, tx_id ASC;
"""

df_sql = pd.read_sql_query(sql_query, conn)
print("=== 👑 SQL 轨道：单表跨行围剿，反欺诈大盘落地 ===")
print(df_sql.to_string(index=False))

=== 👑 SQL 轨道：单表跨行围剿，反欺诈大盘落地 ===
 user_id  tx_id current_location  seconds_since_last_tx
     701   5004         New_York                    300
     702   5006            Tokyo                   1800


In [ ]:
# ====================================================
# 🐼 PANDAS 轨道 - 链式游标位移特征工程（最终完美版）
# ====================================================

# 🛡️ 生产级铁律：前置执行物理熔炼，彻底干掉文本字符串幽灵（dtype('O')）
df_tx['location'] = df_tx['location'].str.strip()
df_tx['transaction_time'] = pd.to_datetime(df_tx['transaction_time'].str.strip())

# 🛰️ 舱内时空排队：这是流式滑窗计算的最高安全防线
df_tx = df_tx.sort_values(by=['user_id', 'transaction_time']).reset_index(drop=True)

df_final_pandas = (
    df_tx
    # 👻 幽灵广播：利用 transform + shift(1) 逆向回溯上一次的地点与时间
    .assign(
        last_location=lambda df: df.groupby('user_id')['location'].transform(lambda x: x.shift(1)),
        last_time=lambda df: df.groupby('user_id')['transaction_time'].transform(lambda x: x.shift(1))
    )
    # ⚔️ 降下反欺诈天网：卡死地点突变，且用 .notna() 物理清除天字第一号行的 NaN 漏洞
    .query("location != last_location and last_location.notna()")
    # ⚡ 终极时空熔炼：两列 Datetime 直接相减，并调用 .dt.total_seconds() 强行榨出数字秒数
    .assign(
        seconds_since_last_tx=lambda df: (df['transaction_time'] - df['last_time']).dt.total_seconds().astype(int)
    )
    # 🧹 按照审计标准收割字段，完成降维交付
    [['user_id', 'tx_id', 'location', 'seconds_since_last_tx']]
    .rename(columns={'location': 'current_location'})
    .sort_values(by=['user_id', 'tx_id'])
    .reset_index(drop=True)
)

print("\n=== 👑 PANDAS 轨道：单表跨行围剿，反欺诈大盘落地 ===")
print(df_final_pandas.to_string(index=False))

=== 👑 PANDAS 轨道：单表跨行围剿，异地盗刷特征大盘落地 ===
 user_id  tx_id current_location  seconds_since_last_tx
     701   5004         New_York                    300
     702   5006            Tokyo                   1800
